In [3]:
!pip uninstall -y dgl torchdata

!pip install -q torch==2.2.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# Install torchdata compatible with torch 2.2
!pip install -q torchdata==0.7.1

# Install DGL for cu121
!pip install dgl -q -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html
!pip install torch_geometric
!pip install numpy
!pip install rdkit
!pip install dgllife

Found existing installation: dgl 2.4.0+cu121
Uninstalling dgl-2.4.0+cu121:
  Successfully uninstalled dgl-2.4.0+cu121
Found existing installation: torchdata 0.7.1
Uninstalling torchdata-0.7.1:
  Successfully uninstalled torchdata-0.7.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires torchdata==0.11.0, but you have torchdata 0.7.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.7 MB/s eta 0:00:00a 0:00:01


In [4]:
import torch
import time
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdchem import HybridizationType
from torch_geometric.datasets import MoleculeNet
import dgl

"""
 TOXCAST PRETRAINING NOTE:
 Reuses the exact same 3D embedding + one-hot FGN augmentation logic as
 the Tox21 pipeline. Only the data source changes — SMILES + raw labels
 are pulled from PyG's MoleculeNet(name="ToxCast") loader instead of a
 manually-downloaded CSV.

 CRITICAL: unlike the original Tox21 script, missing labels are NOT
 filled with 0 here. ToxCast is ~70% missing per assay on average —
 filling with 0 would silently relabel most missing entries as
 "confirmed non-toxic," corrupting both the dynamic weight calculation
 and the loss. Missing values are kept as NaN so the existing NaN-mask
 logic in train_Tox21.py-style loops (is_valid = target_flags == target_flags)
 actually does its job.

 SPARSITY FILTERING: ToxCast has 617 raw assay columns, many with very
 few positive examples in a given split. Columns with fewer than
 MIN_POSITIVES total positive labels across the whole dataset are
 dropped before saving, since they're not learnable and just add noise/
 instability to the dynamic weight formula. The final column count
 (out_features for pretraining) is printed at the end — use that number
 when defining the model's prediction head.
"""

MIN_POSITIVES = 10  # columns with fewer positive labels than this are dropped

# ===========================================================================
#  FUNCTIONAL GROUP NODE (FGN) DETECTION — identical to Tox21 one-hot version
# ===========================================================================
FUNCTIONAL_GROUP_SMARTS = [
    ("benzene",  "c1ccccc1"),
    ("carbonyl", "[CX3]=[OX1]"),
    ("carboxyl",  "[CX3](=O)[OX2H1]"),
    ("hydroxyl",  "[OX2H]"),
    ("amine",     "[NX3;H2,H1;!$(NC=O)]"),
    ("amide",     "[NX3][CX3](=[OX1])"),
    ("nitro",     "[$([NX3](=O)=O),$([NX3+](=O)[O-])]"),
    ("ether",     "[OD2]([#6])[#6]"),
    ("thiol",     "[SX2H]"),
    ("halogen",   "[F,Cl,Br,I]"),
    ("sulfonyl",  "[$([#16X4](=[OX1])=[OX1])]"),
    ("phosphate", "[PX4](=O)"),
]

COMPILED_FG_PATTERNS = [
    (name, Chem.MolFromSmarts(smarts)) for name, smarts in FUNCTIONAL_GROUP_SMARTS
]

FG_NAME_TO_INDEX = {name: idx for idx, (name, _) in enumerate(FUNCTIONAL_GROUP_SMARTS)}
NUM_FG_TYPES = len(FUNCTIONAL_GROUP_SMARTS)  # 12


def detect_functional_groups(mol):
    found = []
    seen_sets = set()
    for fg_name, pattern in COMPILED_FG_PATTERNS:
        if pattern is None:
            continue
        for match in mol.GetSubstructMatches(pattern):
            key = frozenset(match)
            if key not in seen_sets:
                seen_sets.add(key)
                found.append((fg_name, match))
    return found


def build_fgn_augmented_graph(mol, x_tensor, pos_tensor, edge_index_tensor, edge_attr_tensor):
    """
    Identical logic to the Tox21 one-hot version — 7 chemistry features +
    is_fgn flag + 12-dim FG type one-hot = 20 node feature dims total.
    """
    n_atoms = x_tensor.size(0)

    is_fgn_flag   = torch.zeros(n_atoms, 1, dtype=torch.float)
    fg_type_zeros = torch.zeros(n_atoms, NUM_FG_TYPES, dtype=torch.float)
    x_aug = torch.cat([x_tensor, is_fgn_flag, fg_type_zeros], dim=1)  # (N, 20)

    fgn_features_list = []
    fgn_pos_list = []
    new_edges = []
    new_edge_attrs = []

    fg_matches = detect_functional_groups(mol)

    for fg_idx, (fg_name, atom_indices) in enumerate(fg_matches):
        virtual_node_idx = n_atoms + fg_idx

        member_features = x_tensor[list(atom_indices)]
        mean_features   = member_features.mean(dim=0)

        fg_type_onehot = torch.zeros(NUM_FG_TYPES, dtype=torch.float)
        fg_type_onehot[FG_NAME_TO_INDEX[fg_name]] = 1.0

        fgn_row = torch.cat([mean_features, torch.tensor([1.0]), fg_type_onehot])
        fgn_features_list.append(fgn_row)

        member_pos = pos_tensor[list(atom_indices)]
        centroid   = member_pos.mean(dim=0)
        fgn_pos_list.append(centroid)

        for atom_idx in atom_indices:
            new_edges.append([atom_idx,         virtual_node_idx])
            new_edges.append([virtual_node_idx, atom_idx])
            virtual_flag = [0, 0, 0, 0, 1]
            new_edge_attrs.append(virtual_flag)
            new_edge_attrs.append(virtual_flag)

    if fgn_features_list:
        fgn_features = torch.stack(fgn_features_list, dim=0)
        fgn_pos      = torch.stack(fgn_pos_list,      dim=0)
        x_aug   = torch.cat([x_aug,      fgn_features], dim=0)
        pos_aug = torch.cat([pos_tensor, fgn_pos],      dim=0)
    else:
        pos_aug = pos_tensor

    if new_edges:
        new_edge_tensor = torch.tensor(new_edges, dtype=torch.long).t().contiguous()
        edge_aug        = torch.cat([edge_index_tensor, new_edge_tensor], dim=1)
        new_attr_tensor = torch.tensor(new_edge_attrs, dtype=torch.float)
        edge_attr_aug   = torch.cat([edge_attr_tensor, new_attr_tensor], dim=0)
    else:
        edge_aug      = edge_index_tensor
        edge_attr_aug = edge_attr_tensor

    return x_aug, pos_aug, edge_aug, edge_attr_aug


# ===========================================================================
#  MAIN PIPELINE
# ===========================================================================
print("Loading ToxCast via PyG MoleculeNet loader...")
raw_dataset = MoleculeNet(root="/kaggle/working/ToxCast_raw", name="ToxCast")
print(f"PyG loaded {len(raw_dataset)} ToxCast molecules with {raw_dataset[0].y.shape[-1]} raw assay columns.")

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
    "balls. Drake and One Piece goated asl"  # DO NOT DELETE — placeholder for virtual bond column
]

successful_graphs = []  # each entry: (dgl_graph, y_tensor_raw, smiles_string)
failed_count = 0

print(f"Starting 2D-to-3D + One-Hot FGN Pipeline for {len(raw_dataset)} ToxCast molecules.")
start_time = time.time()

for index in range(len(raw_dataset)):
    data_item     = raw_dataset[index]
    smiles_string = data_item.smiles
    labels_raw    = data_item.y.squeeze(0)  # (617,) — NaN preserved for missing entries

    if index % 1000 == 0:
        print(f"Processing Molecule {index} / {len(raw_dataset)}...")

    # --- PHASE 1: 2D Graph Construction ---
    mol = Chem.MolFromSmiles(smiles_string)
    if mol is None:
        failed_count += 1
        continue
    mol = Chem.AddHs(mol)

    # --- PHASE 2: 3D Geometry & Physics ---
    res = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    if res != 0:
        failed_count += 1
        continue

    ff_result = AllChem.MMFFOptimizeMolecule(mol)
    if ff_result != 0:
        failed_count += 1
        continue

    # --- PHASE 3: Geometry Tensor ---
    conformer  = mol.GetConformer()
    pos_tensor = torch.tensor(conformer.GetPositions(), dtype=torch.float)

    # --- PHASE 4: Feature Tensor ---
    atom_features = []
    for atom in mol.GetAtoms():
        hybridization = atom.GetHybridization()
        feature_vector = [
            atom.GetAtomicNum(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic()),
            atom.GetTotalNumHs(),
            1 if hybridization == HybridizationType.SP   else 0,
            1 if hybridization == HybridizationType.SP2  else 0,
            1 if hybridization == HybridizationType.SP3  else 0,
        ]
        atom_features.append(feature_vector)

    x_tensor = torch.tensor(atom_features, dtype=torch.float)

    # --- PHASE 5: Bonds Tensor ---
    edge_indices = []
    edge_attrs   = []
    for bond in mol.GetBonds():
        start_idx    = bond.GetBeginAtomIdx()
        end_idx      = bond.GetEndAtomIdx()
        b_type       = bond.GetBondType()
        bond_feature = [int(b_type == t) for t in BOND_TYPES]
        edge_indices.append([start_idx, end_idx])
        edge_indices.append([end_idx,   start_idx])
        edge_attrs.append(bond_feature)
        edge_attrs.append(bond_feature)

    if len(edge_indices) > 0:
        edge_index_tensor = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr_tensor  = torch.tensor(edge_attrs,   dtype=torch.float)
    else:
        edge_index_tensor = torch.empty((2, 0), dtype=torch.long)
        edge_attr_tensor  = torch.empty((0, len(BOND_TYPES)), dtype=torch.float)

    # --- PHASE 6: ONE-HOT FGN AUGMENTATION ---
    x_aug, pos_aug, edge_index_aug, edge_attr_aug = build_fgn_augmented_graph(
        mol, x_tensor, pos_tensor, edge_index_tensor, edge_attr_tensor
    )

    # --- PHASE 7: Build DGL Graph ---
    src = edge_index_aug[0]
    dst = edge_index_aug[1]

    g = dgl.graph((src, dst), num_nodes=x_aug.size(0))
    g.ndata['x']         = x_aug           # (N+K, 20)
    g.ndata['pos']       = pos_aug          # (N+K, 3)
    g.edata['edge_attr'] = edge_attr_aug    # (E+E_new, 5)

    # NaN preserved — no fillna(0) anywhere in this pipeline
    successful_graphs.append((g, labels_raw, smiles_string))

print(f"\nConversion Completed")
print("-" * 35)
print(f"Successfully generated {len(successful_graphs)} 3D DGL Graphs.")
print(f"Skipped {failed_count} physically impossible molecules.")
print("-" * 35)

# ===========================================================================
#  SPARSITY FILTERING — drop columns with too few positive labels
# ===========================================================================
print("Filtering sparse assay columns...")
all_labels_stacked = torch.stack([item[1] for item in successful_graphs], dim=0)  # (M, 617)

num_pos_per_col = (all_labels_stacked == 1).sum(dim=0)  # NaN != 1, so NaNs don't count
valid_columns   = (num_pos_per_col >= MIN_POSITIVES).nonzero(as_tuple=True)[0]

print(f"Kept {len(valid_columns)} / {all_labels_stacked.size(1)} assay columns "
      f"(threshold: >= {MIN_POSITIVES} positive labels).")

final_graphs = []
for g, labels_raw, smiles_string in successful_graphs:
    labels_filtered = labels_raw[valid_columns]  # (len(valid_columns),)
    final_graphs.append((g, labels_filtered, smiles_string))

print(f"\nFinal ToxCast pretraining out_features = {len(valid_columns)}")
print("⚠️  Use this exact number when defining the model's prediction head for ToxCast pretraining.")

torch.save(final_graphs, "/kaggle/working/toxcast_3d_egnn_dataset_onehot.pt")
torch.save(valid_columns, "/kaggle/working/toxcast_valid_columns.pt")  # keep for reference/debugging

end_time   = time.time()
total_time = end_time - start_time
print(f"Total time: {total_time / 60:.2f} minutes.")
print("Saved successfully as 'toxcast_3d_egnn_dataset_onehot.pt'.")

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
Loading ToxCast via PyG MoleculeNet loader...


Extracting /kaggle/working/ToxCast_raw/toxcast/raw/toxcast_data.csv.gz
Processing...
/usr/local/lib/python3.12/dist-packages/torch_geometric/data/dataset.py:266: UserWarning: Skipping molecule '[F-][B+3]([F-])([F-])[F-].CC[N+]1(C)CCCC1' since it resulted in zero atoms
  self.process()
/usr/local/lib/python3.12/dist-packages/torch_geometric/data/dataset.py:266: UserWarning: Skipping molecule '[NH4+].[NH4+].[Cl-][Pt++]([Cl-])([Cl-])[Cl-]' since it resulted in zero atoms
  self.process()
/usr/local/lib/python3.12/dist-packages/torch_geometric/data/dataset.py:266: UserWarning: Skipping molecule '[Cl-][Pt]1([Cl-])[NH2+]CC[NH2+]1' since it resulted in zero atoms
  self.process()
/usr/local/lib/python3.12/dist-packages/torch_geometric/data/dataset.py:266: UserWarning: Skipping molecule 'FAIL' since it resulted in zero atoms
  self.process()
Done!


PyG loaded 8579 ToxCast molecules with 617 raw assay columns.
Starting 2D-to-3D + One-Hot FGN Pipeline for 8579 ToxCast molecules.
Processing Molecule 0 / 8579...
Processing Molecule 1000 / 8579...
Processing Molecule 2000 / 8579...
Processing Molecule 3000 / 8579...
Processing Molecule 4000 / 8579...
Processing Molecule 5000 / 8579...
Processing Molecule 6000 / 8579...
Processing Molecule 7000 / 8579...
Processing Molecule 8000 / 8579...

Conversion Completed
-----------------------------------
Successfully generated 5625 3D DGL Graphs.
Skipped 2954 physically impossible molecules.
-----------------------------------
Filtering sparse assay columns...
Kept 599 / 617 assay columns (threshold: >= 10 positive labels).

Final ToxCast pretraining out_features = 599
⚠️  Use this exact number when defining the model's prediction head for ToxCast pretraining.
Total time: 13.57 minutes.
Saved successfully as 'toxcast_3d_egnn_dataset_onehot.pt'.
